# Spatial drought analysis

In [ ]:
# general
from pathlib import Path
import geopandas as gpd
import pandas as pd
import yaml 
import os

# plotting
from plotly.subplots import make_subplots
import plotly.graph_objects as go

wd = Path('/home/rooda/OneDrive/Projects/Whatif-Andes')
os.chdir(wd) 

# import config and local module
with open("config.yml") as f:
    cfg = yaml.safe_load(f)

scenarios       = [s["id"] for s in cfg["scenarios"]]
scenarios_names = [s["name"] for s in cfg["scenarios"]]
colors          = [s["color"] for s in cfg["scenarios"]]

## Data

In [ ]:
# drought metrics
df_metrics = pd.read_csv("data/hydro/droughts_hydroatlas_whatif.csv", index_col=0)
df_metrics = df_metrics.pivot(columns="scenario")
df_metrics.columns = [f"{col[0]}_{col[1]}" for col in df_metrics.columns]

# drought extent
df_extent = pd.read_csv("data/hydro/droughts_hydroatlas_area_ts.csv", index_col=["scenario", "date"], parse_dates=["date"])
df_extent = df_extent.rolling(window=90, min_periods=1, center=True).mean()

# shapes/polygons
hydroatlas_shape = gpd.read_file("data/hydro/basins_hydroATLAS_lev08.gpkg")
hydroatlas_shape = hydroatlas_shape[hydroatlas_shape.glacier_area_RGI60 > 0] # only keep basins with glaciers
hydroatlas_shape.gauge_id = hydroatlas_shape.gauge_id.astype(int)
hydroatlas_shape = hydroatlas_shape.set_index("gauge_id")
hydroatlas_shape = hydroatlas_shape.join(df_metrics)

In [ ]:
# latitudinal aggregation of drought metrics

# weighted mean by latitude (weighted by basin area)
hydroatlas_shape['basin_cenlat_rounded'] = hydroatlas_shape['basin_cenlat'].round(0)
weights = hydroatlas_shape.groupby("basin_cenlat_rounded")['basin_area'].sum()
hydroatlas_shape_lat = hydroatlas_shape.drop(columns="geometry").groupby("basin_cenlat_rounded").apply(
    lambda x: (x * x['basin_area'].values[:, None]).sum(axis=0) / x['basin_area'].sum())

# for area
df_extent = df_extent.groupby(level='scenario').mean()
df_extent = df_extent.transpose()
df_extent.index = df_extent.index.astype(int)

for scenario in scenarios:
    df_extent[scenario] = ((df_extent[scenario] / df_extent["historical"]) -1)*100

In [ ]:
hydroatlas_shape.columns

## Plot (MS)

In [ ]:
fig = make_subplots(rows=1, cols=4, horizontal_spacing=0.02, shared_yaxes=True,)

# Panel 1: Severity
for scenario, color in zip(scenarios[::-1], colors[::-1]):
    fig.add_trace(go.Bar(x=hydroatlas_shape_lat["severity_" + scenario], y=hydroatlas_shape_lat.index, 
                         marker_color=color, orientation='h', name = scenarios_names[scenarios.index(scenario)],
                         showlegend=True), 
                  row=1, col=1)

# Panel 2: Min 30-day
for scenario, color in zip(scenarios[::-1], colors[::-1]):
    fig.add_trace(go.Bar(x=hydroatlas_shape_lat["min_30day_" + scenario], y=hydroatlas_shape_lat.index, 
                         marker_color=color, orientation='h',  name = scenarios_names[scenarios.index(scenario)],
                         showlegend=False), 
                  row=1, col=2)

# Panel 3: Day of year minimum
for scenario, color in zip(scenarios[::-1], colors[::-1]):
    fig.add_trace(go.Bar(x=hydroatlas_shape_lat["doy_min_" + scenario], y=hydroatlas_shape_lat.index, 
                         marker_color=color, orientation='h',  name = scenarios_names[scenarios.index(scenario)],
                         showlegend=False),  
                  row=1, col=3)

# Panel 4: Extent
for scenario, color in zip(scenarios[::-1], colors[::-1]):
    fig.add_trace(go.Bar(x=df_extent[scenario], y=df_extent.index - 1, 
                         marker_color=color, orientation='h',  name = scenarios_names[scenarios.index(scenario)],
                         showlegend=False), 
                  row=1, col=4)

# Add panel labels (a, b, c, d) to lower left of each plot
labels = ['a)', 'b)', 'c)', 'd)']
x_positions = [0.21, 0.46, 0.72, 0.97]  # Adjusted for 4 panels with horizontal_spacing=0.02
for i, (label, x_pos) in enumerate(zip(labels, x_positions)):
    fig.add_annotation(
        text=label,
        x=x_pos,
        y=0.995,
        xref='paper',
        yref='paper',
        showarrow=False,
        font=dict(size=14, family='Arial', color='black'),
        xanchor='left',
        yanchor='top'
    )



# Update settings for each subplot
fig.update_xaxes(ticks="outside", griddash="dot", zeroline=False)
fig.update_yaxes(ticks="outside", griddash="dot", range = [-56, -7], ticksuffix = "ºS", zeroline=False)
fig.update_xaxes(title_text="Δ Severity (%)", title_standoff=5, row=1, col=1)
fig.update_xaxes(title_text="Δ Min 30-day flow (%)", title_standoff=5, row=1, col=2)
fig.update_xaxes(title_text="Δ DOY minimum flow (days)", title_standoff=5, row=1, col=3)
fig.update_xaxes(title_text="Δ Extent (%)", title_standoff=5, row=1, col=4)

# Add vertical line on the left
fig.add_shape(
    type="line",
    x0=-0.09, x1=-0.09,
    y0=0, y1=1.0,
    xref="paper", yref="paper",
    line=dict(width=1)
)

# Add horizontal lines at region boundaries
for boundary_lat in [-7, -20, -30, -35, -45, -52, -56]:
    y_pos = (boundary_lat - (-56)) / ((-7) - (-56))
    fig.add_shape(
        type="line",
        x0=-0.10, x1=-0.08,
        y0=y_pos, y1=y_pos,
        xref="paper", yref="paper",
        line=dict(width=1)
    )

# Add region annotations on the vertical line
for region, (lat_min, lat_max) in cfg["regions"].items():
    # Calculate midpoint of the region in latitude

    mid_lat = (lat_min + lat_max) / 2
    y_pos = (mid_lat - (-56)) / ((-7) - (-56))
    
    fig.add_annotation(
        text=region.replace(' ', '<br>'),
        bgcolor="white", 
        x=-0.09,
        y=y_pos,
        xref="paper",
        yref="paper",
        showarrow=False,
        font=dict(size=13, family="Arial"),
        xanchor="center",
        yanchor="middle",
    )

fig.update_layout(width=900, height=750, 
                  barmode = "overlay",
                  legend =dict(orientation="h", yanchor="bottom", y=1, xanchor="right", x=1),
                  margin=dict(l=110, r=10, b=10, t=0), 
                  font_family="Arial", font_size=12)

fig.write_image("figures/fig_whatif_droughts_latitudinal.svg", width=900, height=750)
fig.show()

## Plot (slides)

In [ ]:
fig = make_subplots(rows=1, cols=5, horizontal_spacing=0.015, shared_yaxes=True,)

# Panel 1: Severity
for scenario, color in zip(scenarios[::-1], colors[::-1]):
    fig.add_trace(go.Bar(x=hydroatlas_shape_lat["severity_" + scenario], y=hydroatlas_shape_lat.index, 
                         marker_color=color, orientation='h', name = scenarios_names[scenarios.index(scenario)],
                         showlegend=True), 
                  row=1, col=1)
fig.update_xaxes(title_text="Δ Severity (%)", title_standoff=5, row=1, col=1)

# Panel 2: Min 30-day
for scenario, color in zip(scenarios[::-1], colors[::-1]):
    fig.add_trace(go.Bar(x=hydroatlas_shape_lat["min_30day_" + scenario], y=hydroatlas_shape_lat.index, 
                         marker_color=color, orientation='h',  name = scenarios_names[scenarios.index(scenario)],
                         showlegend=False), 
                  row=1, col=2)
fig.update_xaxes(title_text="Δ Min 30-day flow (%)", title_standoff=5, row=1, col=2)

# Panel 3: (?)
for scenario, color in zip(scenarios[::-1], colors[::-1]):
    fig.add_trace(go.Bar(x=hydroatlas_shape_lat["total_duration_" + scenario], y=hydroatlas_shape_lat.index, 
                         marker_color=color, orientation='h',  name = scenarios_names[scenarios.index(scenario)],
                         showlegend=False),  
                  row=1, col=3)
fig.update_xaxes(title_text="Δ Total duration (%)", title_standoff=5, row=1, col=3)

# Panel 4: Day of year minimum
for scenario, color in zip(scenarios[::-1], colors[::-1]):
    fig.add_trace(go.Bar(x=hydroatlas_shape_lat["doy_min_" + scenario], y=hydroatlas_shape_lat.index, 
                         marker_color=color, orientation='h',  name = scenarios_names[scenarios.index(scenario)],
                         showlegend=False),  
                  row=1, col=4)
fig.update_xaxes(title_text="Δ DOY minimum flow (days)", title_standoff=5, row=1, col=4)

# Panel 5: Intensity
for scenario, color in zip(scenarios[::-1], colors[::-1]):
    fig.add_trace(go.Bar(x=hydroatlas_shape_lat["intensity_" + scenario], y=hydroatlas_shape_lat.index, 
                         marker_color=color, orientation='h',  name = scenarios_names[scenarios.index(scenario)],
                         showlegend=False),  
                  row=1, col=5)
fig.update_xaxes(title_text="Δ Intensity (%)", title_standoff=5, row=1, col=5)


# Update settings for each subplot
fig.update_xaxes(ticks="outside", griddash="dot", zeroline=False)
fig.update_yaxes(ticks="outside", griddash="dot", range = [-56, -7], ticksuffix = "ºS", zeroline=False)


# Add vertical line on the left
fig.add_shape(
    type="line",
    x0=-0.07, x1=-0.07,
    y0=0, y1=1.0,
    xref="paper", yref="paper",
    line=dict(width=1)
)

# Add horizontal lines at region boundaries
for boundary_lat in [-7, -20, -30, -35, -45, -52, -56]:
    y_pos = (boundary_lat - (-56)) / ((-7) - (-56))
    fig.add_shape(
        type="line",
        x0=-0.08, x1=-0.06,
        y0=y_pos, y1=y_pos,
        xref="paper", yref="paper",
        line=dict(width=1)
    )

# Add region annotations on the vertical line
for region, (lat_min, lat_max) in cfg["regions"].items():
    # Calculate midpoint of the region in latitude

    mid_lat = (lat_min + lat_max) / 2
    y_pos = (mid_lat - (-56)) / ((-7) - (-56))
    
    fig.add_annotation(
        text=region.replace(' ', '<br>'),
        bgcolor="white", 
        x=-0.07,
        y=y_pos,
        xref="paper",
        yref="paper",
        showarrow=False,
        font=dict(size=13, family="Arial"),
        xanchor="center",
        yanchor="middle",
    )

fig.update_layout(width=1100, height=700, 
                  barmode = "overlay",
                  legend =dict(title="Scenarios:", orientation="h", yanchor="bottom", y=1, xanchor="right", x=1),
                  margin=dict(l=110, r=10, b=10, t=0), 
                  font_family="Arial", font_size=12)

fig.write_image("figures/fig_whatif_droughts_latitudinal_slides.svg", width=1100, height=700)
fig.show()